# YouTube Comment Sentiment Analyzer (LSTM)
**bold text**
**What we'll build:** paste a YouTube link (e.g. a movie trailer) → we download its comments → an LSTM neural network labels each comment as *positive* or *negative* → we print a final verdict.

**How to use this notebook:** run the cells from top to bottom (click ▶ or press `Shift+Enter`). Read the text above each cell to learn what it does.

**Tip:** Go to `Runtime → Change runtime type → T4 GPU` to make training faster.

**The plan**
1. Install & import libraries
2. Load a labelled dataset (IMDB movie reviews)
3. Prepare the text for the neural network
4. Build & train the LSTM
5. Take a YouTube URL, download comments, predict, and give a verdict

## Step 1: Install and import libraries

- `youtube-comment-downloader` fetches comments (no API key needed).
- `tensorflow` / `keras` is used for the LSTM (already installed in Colab).
- `re` cleans text, `numpy` handles numbers.

In [1]:
!pip install -q youtube-comment-downloader

In [2]:
import re
import numpy as np
from itertools import islice
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR

print("TensorFlow/Keras ready ✅")

TensorFlow/Keras ready ✅


## Step 2: Load the training data

A neural network learns from examples. We use the **IMDB dataset**: 50,000 movie reviews, each labelled `1` (positive) or `0` (negative).

Two settings to know:
- `VOCAB_SIZE = 10000` → we keep only the 10,000 most common words.
- `MAX_LEN = 200` → every review is cut or padded to exactly 200 words (networks need equal-sized inputs).

The dataset comes **already converted to numbers**: each word is replaced by a number (e.g. "great" → 87).

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Training reviews:", len(x_train))
print("Test reviews:", len(x_test))
print("First review (as numbers):", x_train[0][:15], "...")
print("Its label (1 = positive, 0 = negative):", y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training reviews: 25000
Test reviews: 25000
First review (as numbers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4] ...
Its label (1 = positive, 0 = negative): 1


## Step 3: Make all reviews the same length

`pad_sequences` adds zeros to short reviews and trims long ones, so every review has exactly `MAX_LEN` numbers.

In [4]:
x_train = keras.utils.pad_sequences(x_train, maxlen=MAX_LEN)
x_test = keras.utils.pad_sequences(x_test, maxlen=MAX_LEN)

print("Shape of training data:", x_train.shape)

Shape of training data: (25000, 200)


## Step 4: Build the LSTM model

Layers, in order:
1. **Embedding**: turns each word number into a small list of 64 values that capture its meaning.
2. **LSTM**: reads the sentence word by word, *remembering* important earlier words (e.g. "not" in "not good").
3. **Dropout**: randomly switches off some neurons while training so the model doesn't just memorize.
4. **Dense (sigmoid)**: outputs one number from `0` (negative) to `1` (positive).

In [5]:
model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN,)),
    keras.layers.Embedding(VOCAB_SIZE, 64),
    keras.layers.LSTM(64),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,089 (2.57 MB)

 Trainable params: 673,089 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

## Step 5: Train the model

One **epoch** = the model sees all training reviews once. We run 3 epochs. `validation_split=0.2` keeps 20% of the data aside to check progress on reviews it hasn't trained on.

(Takes a few minutes on CPU, faster on GPU.)

In [6]:
history = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.2)

Epoch 1/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.6930 - loss: 0.5708 - val_accuracy: 0.8366 - val_loss: 0.3761
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8767 - loss: 0.3007 - val_accuracy: 0.8710 - val_loss: 0.3060
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9216 - loss: 0.2103 - val_accuracy: 0.8756 - val_loss: 0.3308


## Step 6: Check accuracy on unseen reviews

The test set was never used in training, so this tells us how well the model really performs (expect roughly 85%).

In [7]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy:.2%}")

Test accuracy: 86.34%


## Step 7: Helper functions to handle new text

YouTube comments are plain text, but our model only understands numbers. So we need to:
1. **Clean** the text (lowercase, remove links, emojis, punctuation).
2. **Convert** each word to its IMDB number, using the same numbering the model was trained on.

Special numbers in the IMDB data: `1` = start of text, `2` = unknown word. All real word numbers are shifted by `+3`.

In [8]:
word_index = imdb.get_word_index()   # dictionary: word -> number

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)     # remove links
    text = re.sub(r"[^a-z' ]", " ", text)    # keep only letters and apostrophes
    return text

def encode_comments(comments):
    all_sequences = []
    for comment in comments:
        sequence = [1]                        # start marker
        for word in clean_text(comment).split():
            number = word_index.get(word)
            if number is not None and number + 3 < VOCAB_SIZE:
                sequence.append(number + 3)
            else:
                sequence.append(2)            # unknown word
        all_sequences.append(sequence)
    return keras.utils.pad_sequences(all_sequences, maxlen=MAX_LEN)

# Quick test on two made-up sentences
test_comments = ["This trailer looks amazing, I can't wait!", "Terrible. Boring and a waste of time."]
scores = model.predict(encode_comments(test_comments), verbose=0).flatten()
for comment, score in zip(test_comments, scores):
    print(f"{score:.2f}  ->  {comment}")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
0.85  ->  This trailer looks amazing, I can't wait!
0.04  ->  Terrible. Boring and a waste of time.


## Step 8: Enter a YouTube URL

Run the cell below, then **paste your link into the box that appears** and press Enter.
Example: `https://www.youtube.com/watch?v=...`

In [9]:
url = input("Paste a YouTube URL: ").strip()
MAX_COMMENTS = 300   # change this to analyze more or fewer comments

Paste a YouTube URL: https://www.youtube.com/watch?v=LM4ByXufv2w&pp=ygUNc2NlbmUgdHJhaWxlcg%3D%3D


## Step 9: Download the comments

We fetch the most popular comments, up to `MAX_COMMENTS`.

In [10]:
downloader = YoutubeCommentDownloader()
comment_generator = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
comments = [c["text"] for c in islice(comment_generator, MAX_COMMENTS)]

print(f"Downloaded {len(comments)} comments")
print("Example:", comments[0] if comments else "No comments found")

Downloaded 300 comments
Example: The Malayalam makers’ touch was evident in every sequence of the trailer. The feel, the treatment, and the overall making were completely different from typical Tamil films. And seeing Naslen’s growth as an actor is something else!

And finally, we got some relief from Anirudh! 🤣


## Step 10: Predict sentiment and give the verdict

For every comment the model gives a score between 0 and 1. We call it **positive if the score is 0.5 or higher**, otherwise **negative**. Then we count the percentages.

**Verdict rule:** 60%+ positive → positively rated · 40% or less → negatively rated · in between → mixed.

In [11]:
scores = model.predict(encode_comments(comments), verbose=0).flatten()

positive_count = int((scores >= 0.5).sum())
negative_count = len(scores) - positive_count
positive_pct = positive_count / len(scores) * 100
negative_pct = 100 - positive_pct

print(f"Comments analyzed: {len(scores)}")
print(f"Positive: {positive_pct:.1f}%")
print(f"Negative: {negative_pct:.1f}%")
print()

if positive_pct >= 60:
    print("VERDICT: This movie is POSITIVELY rated by the audience!")
elif positive_pct <= 40:
    print("VERDICT: This movie is NEGATIVELY rated by the audience.")
else:
    print("VERDICT: Audience opinion is MIXED.")

Comments analyzed: 300
Positive: 86.3%
Negative: 13.7%

VERDICT: This movie is POSITIVELY rated by the audience!
